# Espacio por caso e integridad (P0-B2)
Un caso = un dataset. Crea la carpeta del caso, ingiere, abre el ledger y verifica la cadena de custodia completa.

In [ ]:
%load_ext autoreload
%autoreload 2
from dfir_copilot.cases import CaseWorkspace
from dfir_copilot.detectors.roles import resolve_roles

CASO = "IDOR-INVOICES-2020Q4-B"      # nombre NUEVO: el caso anterior sigue intacto en data/ledger
ORIGINAL = "/workspace/data/raw/three_months.csv"
MAPPING = "web_access_meli"

ws = CaseWorkspace.create(CASO, root="/workspace/data/cases", analyst="eder", lang="es")
ws.add_raw(ORIGINAL, link=True)       # enlace simbólico: no duplica 1 GB; el hash se verifica igual
print(ws.dir)

In [ ]:
%%time
manifest = ws.ingest(ORIGINAL, MAPPING)
print(manifest["output"]["rows"], "filas |", manifest["time_range_utc"])
print("roles del mapping:", manifest["roles"])

In [ ]:
engine = ws.engine()
ledger = ws.ledger(engine)
roles = resolve_roles(engine, lang="es")
print(roles.actor, "/", roles.resource, roles.sources)
for n in roles.notes:
    print(" -", n)

In [ ]:
informe = ws.verify()
print(informe.render("es"))
print("\nCADENA DE CUSTODIA ÍNTEGRA:", informe.ok)

In [ ]:
print("sellado:", ws.sealed)
try:
    ws.ingest(ORIGINAL, MAPPING)
except Exception as e:
    print(type(e).__name__, "->", e)

In [ ]:
print(ledger.summary())
print("casos:", CaseWorkspace.list_cases("/workspace/data/cases"))